In [82]:
import numpy as np
from nltk.corpus import brown
from sklearn.preprocessing import OneHotEncoder

vocab = np.unique(brown.words()).reshape(-1, 1)

""" Hyperparameters """
vocab_size = len(vocab)
activations = 100

# print("Vocabulary Size: ", vocab_size)
# print("Vocabulary: ", vocab)

### Creating word encoding

In [83]:
enc = OneHotEncoder(handle_unknown='ignore') # one hot encoding for the entire vocabulary 
enc.fit(vocab)

word = 'and'
categories = enc.categories_
test_enc = enc.transform([[word]]).toarray()
test_word = enc.inverse_transform(test_enc)

# print("Categories: ", categories)
# print(f"One-Hot Vector for '{word}': ", test_enc)
# print("One-Hot Vector Shape: ", test_enc.shape)
# print("Mapping One-Hot Vector to Word: ", test_word)

In [84]:
from sklearn.model_selection import train_test_split

window = 2
words = brown.words()

X, y = [], []
for i in range(len(words)):
    center, context = [words[i]], []

    for j in range(window): # words before the center
        if (i + j) - window >= 0:
            context.append(words[(i + j) - window])
    
    for j in range(window): # words after the center
        if i + j + 1 < len(words):
            context.append(words[i + j + 1])
    
    X.append(center)
    y.append(context)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [85]:
for i in range(20):
    print(X_train[i], y_train[i])

['job'] ['a', 'full-time', 'of', 'cow']
["''"] ['live', 'in', '.', 'Oso']
[','] ['a', 'confused', 'soaked', 'and']
['but'] ['cast', ',', 'every', 'member']
['and'] ['the', 'needs', 'aspirations', 'of']
['no'] ['they', 'had', 'guide', 'for']
['for'] ['natural', 'method', 'the', 'control']
['the'] ['freedom', 'in', 'schools', 'here']
['Harvard'] ['conscience', 'of', 'University', 'for']
['clouds'] ['the', 'new', 'of', 'radioactive']
['undergraduates'] ['to', 'American', '.', 'In']
['Today'] ['lived', '.', 'he', 'is']
['goitrogens'] ['naturally', 'occurring', 'may', 'play']
['that'] ['a', 'manner', 'was', 'reserved']
['friends'] ['your', 'best', 'would', 'think']
['are'] ['that', 'many', 'being', 'forced']
['be'] ['he', 'would', 'instructing', 'his']
['from'] ['the', 'signal', "Dowling's", 'car']
['in'] ['guess', 'that', 'a', 'correct']
['embraces'] [',', 'which', 'all', 'the']


In [86]:
import torch
import torch.nn as nn

class Skipgram(nn.Module):
    def __init__(self):
        super().__init__() # neural network inherits from nn.Module

        """ skipgram network layers """
        self.l1 = nn.Linear(vocab_size, activations)
        self.relu = nn.ReLU()
        self.l2 = nn.Linear(vocab_size, activations) # nn.CrossEntropyLoss will apply softmax

    """ forward pass of data """
    def forward(self, x):
        x = self.l1(x)
        x = self.relu(x)
        logits = self.l2(x)
        return logits
    
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using mps device
